# Public benchmark routing probes

Full current SWE-bench and active BFCL V4 data are downloaded locally. Only ten official ToolBench examples are available; its full archive is still blocked.

**These datasets do not contain gold labels for our six coding-agent routes.** The exported labels are automatic, unreviewed proposals. Scores below measure agreement with those proposals, not native SWE-bench resolution or tool-calling accuracy. Ambiguous inputs, unreconstructed environments and ToolBench samples are excluded from scored probes.

This notebook defaults to local inspection and a 20-record rules smoke test. Azure calls require an explicit opt-in. No benchmark code, tools, APIs or native evaluation scripts are executed.

In [ ]:
import json
import random
import sys
from pathlib import Path
from uuid import uuid4

ROOT = next((parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'routing_lab').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project workspace.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from routing_lab.benchmark_data import jsonl, verify_snapshot
from routing_lab.backends import RulesBackend, questions, state
from routing_lab.data import load_contracts, load_records
from routing_lab.evaluation import run_experiment, sha256

RAW = ROOT / 'data' / 'upstream' / 'public-benchmarks-20260930'
BUNDLE = ROOT / 'data' / 'public-routing' / 'public-benchmarks-20260930'
downloads = verify_snapshot(RAW)
manifest = json.loads((BUNDLE / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'complete_unreviewed_probes'
assert sha256(RAW / 'download-manifest.json') == manifest['download_manifest_sha256']
for name, entry in manifest['files'].items():
    assert sha256(BUNDLE / name) == entry['sha256'], f'Changed artifact: {name}'
records = load_records(BUNDLE / 'endpoint-probes.jsonl')
contracts = load_contracts(ROOT / 'contracts' / 'agents.json')
print('Verified source files:', len(downloads['files']))
print('Candidate counts:', manifest['source_counts'])
print('Draft route counts:', manifest['draft_route_counts'])
print('Endpoint-character-compatible probes:', len(records))
print('Excluded:', manifest['exclusion_reason_counts'])

In [ ]:
print(json.dumps(manifest['category_counts'], indent=2))
print('SWE dev/test repository overlap:', manifest['swe_native_dev_test_repository_overlap'])
print('ToolBench full archive:', downloads['toolbench_full_status'])
for limitation in manifest['limitations']:
    print('-', limitation)

## One exact model input

The payload contains only the issue/request, allowed decision-time context and agent criteria. Proposed labels, native solutions and future conversation turns are not sent. SWE-bench repository content has not been fetched; its labels describe issue intent rather than proving that a downstream agent has enough context to solve it.

In [ ]:
example = next(record for record in records if record.tags[0] == 'swe-bench' and 'dev' in record.tags)
payload = {'state': state(example, 'context'), 'questions': questions(contracts)}
print(json.dumps(payload, indent=2, ensure_ascii=False))
target = next(row for row in jsonl(BUNDLE / 'native-targets.jsonl') if row['id'] == example.id)
print('\nStored separately, NOT model inputs:', list(target['native_targets']))
print('Unreviewed proposed route, NOT sent:', example.expected_route)
print('State characters:', len(payload['state']))

## Small, reproducible development sample

Select up to ten SWE native-dev records and ten BFCL evaluation records, with no repeated query groups. BFCL records inspected here are development-exposed and must not later be claimed as untouched held-out evaluation. The larger bundle preserves native splits; it does not invent a training split.

The rules smoke test validates the pipeline only. Its lexical patterns overlap the automatic label policy, so agreement is not independent evidence of routing quality.

In [ ]:
sample = []
seen_groups = set()
rng = random.Random(42)
for source in ('swe-bench', 'bfcl'):
    pool = [record for record in records if record.tags[0] == source and (source != 'swe-bench' or 'dev' in record.tags)]
    rng.shuffle(pool)
    selected = 0
    for record in pool:
        if record.group_id in seen_groups:
            continue
        sample.append(record)
        seen_groups.add(record.group_id)
        selected += 1
        if selected == 10:
            break
assert sample and len({record.group_id for record in sample}) == len(sample)
sources = {
    'records': BUNDLE / 'endpoint-probes.jsonl',
    'dataset_manifest': BUNDLE / 'manifest.json',
    'contracts': ROOT / 'contracts' / 'agents.json',
    'notebook': ROOT / 'notebooks' / '04_public_benchmarks.ipynb',
}
print('Selected', len(sample), 'development probes; no Azure request has been made.')

In [ ]:
local_output = ROOT / 'outputs' / f'public-rules-smoke-{uuid4().hex[:10]}'
local_result = run_experiment(RulesBackend(), sample, local_output, probe=True, source_paths=sources)
print('Local smoke-test artifacts:', local_output)
print('Agreement with weak labels, not verified correctness:', local_result['strict_accuracy'])

## Optional Azure Clef probe

Set `RUN_AZURE = True` after reviewing the selected inputs and upstream usage terms. This sends only the selected public inputs to your existing Azure endpoint and makes up to 20 billed inference requests. Sign in with Azure CLI first. No new Azure resource or local model download is needed.

A state fitting the endpoint's character limit can still exceed Clef's token budget. Check state truncation, per-source slices and latency before interpreting agreement. This notebook does not tune prompts on the remaining SWE native-test set.

In [ ]:
RUN_AZURE = False
if RUN_AZURE:
    from routing_lab.endpoint import ClefEndpointBackend
    connection = json.loads((ROOT / 'azure' / 'online' / 'connection.json').read_text(encoding='utf-8'))
    backend = ClefEndpointBackend(contracts, url=connection['scoring_uri'], context_mode='context')
    azure_output = ROOT / 'outputs' / f'public-clef-probe-{uuid4().hex[:10]}'
    result = run_experiment(backend, sample, azure_output, probe=True, source_paths=sources)
    print(json.dumps(result, indent=2))
    print('Saved predictions:', azure_output / 'predictions.jsonl')
else:
    print('Azure evaluation disabled. No model requests were sent.')